In [1]:
!pip install langchain
!pip install torch
!pip install sentence_transformers
!pip install faiss-cpu
!pip install huggingface-hub
!pip install pypdf
!pip -q install accelerate
!pip install llama-cpp-python
!pip -q install git+https://github.com/huggingface/transformers
!pip install qdrant_client

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 14.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 5.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.4/49.4 kB 7.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.0/86.0 kB 2.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 65.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 62.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.0/302.0 kB 33.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 76.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 88.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.0/295.0 kB 36.6 MB/s eta 0:00:00
  Created wheel for sentence_transformers: filename=sentence_transformers-2.2.2-py3-none-any.whl size=125923 sha256=c73437d795dd4b9488b72d41cc4425d92b6258c186

In [2]:
from qdrant_client import QdrantClient
from qdrant_client.http import models
from qdrant_client.http.models import CollectionStatus

In [13]:
client = QdrantClient(host="localhost", port=6333)
client

In [4]:
from langchain.chains import RetrievalQA
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.llms import LlamaCpp
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import FAISS
from langchain.document_loaders import PyPDFDirectoryLoader

In [5]:
#load pdf files
loader = PyPDFDirectoryLoader("/content/")
data = loader.load()

In [6]:
print(data)

[Document(page_content="Title:\nThe\nImpact\nof\nRenewable\nEnergy\non\nthe\nGlobal\nEnergy\nLandscape\nThe\nworld\nis\nundergoing\na\nsignificant\ntransformation\nin\nits\nenergy\nlandscape,\nwith\nthe\nincreasing\nadoption\nof\nrenewable\nenergy\nsources.\nThis\nshift\nis\ndriven\nby\nthe\ngrowing\nconcerns\nabout\nclimate\nchange,\nthe\nneed\nto\nreduce\ngreenhouse\ngas\nemissions,\nand\nthe\ndesire\nfor\nenergy\nindependence.\nRenewable\nenergy\ntechnologies,\nsuch\nas\nsolar,\nwind,\nand\nhydropower,\nhave\ngained\nmomentum\nand\nare\nchanging\nthe\nway\nwe\ngenerate\nand\nconsume\nenergy.\nRenewable\nEnergy\nGrowth\nRenewable\nenergy\nsources\nhave\nwitnessed\nsubstantial\ngrowth\nin\nrecent\nyears.\nSolar\nphotovoltaic\n(PV)\nand\nwind\npower,\nin\nparticular,\nhave\nbecome\nmore\naffordable\nand\nefficient,\nmaking\nthem\nattractive\noptions\nfor\nboth\ndeveloped\nand\ndeveloping\nnations.\nGovernments\nand\ncorporations\nare\ninvesting\nin\nlarge-scale\nrenewable\nenergy\nproj

In [16]:
# #create new cluseter in qdrant
record=0



connection = QdrantClient(
    url=os.environ["QDRANT_URL"],
    api_key=os.environ["QDRANT_API_KEY"],
)

connection.recreate_collection(
    collection_name="embeddings_chatbot",
    vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE),
)
print("Create collection reponse:", connection)

info = connection.get_collection(collection_name="embeddings_chatbot")

print("Collection info:", info)
for get_info in info:
  print(get_info)


Create collection reponse: <qdrant_client.qdrant_client.QdrantClient object at 0x7d7f801e7490>
Collection info: status=<CollectionStatus.GREEN: 'green'> optimizer_status=<OptimizersStatusOneOf.OK: 'ok'> vectors_count=0 indexed_vectors_count=0 points_count=0 segments_count=2 config=CollectionConfig(params=CollectionParams(vectors=VectorParams(size=1536, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None), shard_number=1, replication_factor=1, write_consistency_factor=1, read_fan_out_factor=None, on_disk_payload=True), hnsw_config=HnswConfig(m=16, ef_construct=100, full_scan_threshold=10000, max_indexing_threads=0, on_disk=False, payload_m=None), optimizer_config=OptimizersConfig(deleted_threshold=0.2, vacuum_min_vector_number=1000, default_segment_number=0, max_segment_size=None, memmap_threshold=None, indexing_threshold=20000, flush_interval_sec=5, max_optimization_threads=1), wal_config=WalConfig(wal_capacity_mb=32, wal_segments_ahead=0), qu

In [17]:
#Step 05: Split the Extracted Data into Text Chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=10000, chunk_overlap=20)

text_chunks = text_splitter.split_documents(data)


In [18]:
len(text_chunks)

2

In [20]:
#get the third chunk
text_chunks[1]

Document(page_content="Title:\nThe\nImpact\nof\nArtificial\nIntelligence\nin\nHealthcare\nArtificial\nIntelligence\n(AI)\nis\nrevolutionizing\nthe\nhealthcare\nindustry\nby\nenhancing\npatient\ncare,\noptimizing\nadministrative\nprocesses,\nand\nfacilitating\nmedical\nresearch.\nFrom\npredictive\nanalytics\nto\nrobotic\nsurgery,\nAI\ntechnologies\nhave\nthe\npotential\nto\ntransform\nthe\nway\nhealthcare\nis\ndelivered\nand\nimprove\npatient\noutcomes.\nAI-Powered\nDiagnostics\nOne\nof\nthe\nmost\nsignificant\nimpacts\nof\nAI\nin\nhealthcare\nis\nits\nability\nto\nassist\nin\ndiagnosis.\nMachine\nlearning\nalgorithms\ncan\nanalyze\nmedical\ndata,\nsuch\nas\nimaging\nscans\nand\npatient\nrecords,\nto\ndetect\npatterns\nand\nmake\nmore\naccurate\ndiagnoses.\nThis\ncan\nlead\nto\nearly\ndetection\nof\ndiseases\nand\nbetter\ntreatment\nplanning.\nTelemedicine\nand\nVirtual\nHealth\nAssistants\nTelemedicine\nhas\nseen\na\nrapid\nrise\nwith\nthe\nhelp\nof\nAI-powered\nvirtual\nhealth\nassist

In [21]:
#Step 06:Downlaod the Embeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


In [22]:
#Step 08: Create Embeddings for each of the Text Chunk
vector_store = FAISS.from_documents(text_chunks, embedding=embeddings)

In [44]:


# #funcrion for read data from pdf
def read_data_from_pdf():
    pdf_path = '/content/Title_ The Impact of Artificial Intelligence in Healthcare.pdf'
    text = ""  # for storing the extracted text

    with open(pdf_path, 'rb') as file:
        pdf_reader = PdfReader(file)

        for page in pdf_reader.pages:
            text += page.extract_text()

    return text



In [30]:
def get_text_chunks(text):
    text_splitter = CharacterTextSplitter(
        separator="\n",
        chunk_size=1000,
        chunk_overlap=200,
        length_function=len
    )
    chunks = text_splitter.split_text(text)
    return chunks

In [42]:
!pip install PyPDF2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 3.1 MB/s eta 0:00:00


In [43]:
from PyPDF2 import PdfReader
from langchain.text_splitter import CharacterTextSplitter
from langchain.vectorstores import Qdrant
from langchain.embeddings import OpenAIEmbeddings
from qdrant_client import QdrantClient,models
from qdrant_client.http.models import PointStruct
import os
import uuid

In [32]:


# #code for convert chunks into embeddings
def get_embedding(text_chunks, model_id="sentence-transformers/all-MiniLM-L6-v2"):
    points = []
    for idx, chunk in enumerate(text_chunks):
        response = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
        embeddings = response['data'][0]['embedding']
        point_id = str(uuid.uuid4())  # Generate a unique ID for the point

        points.append(PointStruct(id=point_id, vector=embeddings, payload={"text": chunk}))

    return points






In [33]:


# # code for insert data into qdrant database

def insert_data(get_points):
    operation_info = connection.upsert(
    collection_name="embeddings_chatbot",
    wait=True,
    points=get_points
)




In [37]:
!pip install openai
import openai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 1.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
llmx 0.0.15a0 requires cohere, which is not installed.
llmx 0.0.15a0 requires tiktoken, which is not installed.


In [38]:

openai.api_key = os.environ["OPENAI_API_KEY"]

In [39]:
# code for searching
def create_answer_with_context(query):
    response = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    embeddings = response['data'][0]['embedding']

    search_result = connection.search(
        collection_name="embeddings_chatbot",
        query_vector=embeddings,
        limit=1
    )

    prompt = "Context:\n"
    for result in search_result:
        prompt += result.payload['text'] + "\n---\n"
    prompt += "Question:" + query + "\n---\n" + "Answer:"

    print("----PROMPT START----")
    print(":", prompt)
    print("----PROMPT END----")

    completion = openai.ChatCompletion.create(
        model="gpt-3.5-turbo",
        messages=[
            {"role": "user", "content": prompt}
        ]
        )

    return completion.choices[0].message.content


In [40]:
def main():
  get_raw_text=read_data_from_pdf()
  chunks=get_text_chunks(get_raw_text)
  vectors=get_embedding(chunks)

  insert_data(vectors)
  question="what did the dog see at the bakery?"
  answer=create_answer_with_context(question)
  print(answer)


In [45]:
if __name__ == '__main__':
    main()


TypeError: ignored